* 행렬분해 로직을 새로운 함수로 만듦

In [10]:
from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
    error = 0
    full_pred_matrix = np.dot(P, Q.T)
    #실제 R행렬에서 널이 아닌 값의 위치 인덱스 추출 -> 실제 R행렬과 예측 행렬의 RMSE추출
    x_non_zero_idx = [non_zero[0] for non_zero in non_zeros]
    y_non_zero_idx = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero_idx, y_non_zero_idx]
    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_idx, y_non_zero_idx]
    mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
    rmse = np.sqrt(mse)
    
    return rmse

def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda=0.01):
    num_users, num_items = R.shape
    #P, Q 행렬의 크기 지정, 정규 분포 가진 임의의 값으로 입력
    np.random.seed(1)
    P = np.random.normal(scale=1./K, size=(num_users, K))
    Q = np.random.normal(scale=1./K, size=(num_items, K))
    
    prev_rmse = 10000
    break_count = 0
    
    #R>0인 행, 열 위치, 값을 non_zeros 리스트 객체에 저장
    non_zeros = [(i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j] > 0]
    
    #SGD 기법으로 P, Q 매트릭스 업데이트
    for step in range(steps):
        for i, j, r in non_zeros:
            eij = r - np.dot(P[i, :], Q[j, :].T) #오류값
            #규제 반영한 SGD 업데이트 공식 적용
            P[i, :] = P[i, :] + learning_rate * (eij * Q[j, :] - r_lambda * P[i, :])
            Q[j, :] = Q[j, :] + learning_rate * (eij * P[i, :] - r_lambda * Q[j, :])
        rmse = get_rmse(R, P, Q, non_zeros)
        if(step%50)==0:
            print('### iteration step : ', step, "  rmse : ", rmse)
            
    return P, Q

In [6]:
### 영화 평점 행렬 데이터 새로 로딩하고 사용자-아이템 평점 행렬로 만들기

import pandas as pd
import numpy as np

movies = pd.read_csv('../movielens/movies.csv')
ratings = pd.read_csv('../movielens/ratings.csv')
ratings = ratings[['userId', 'movieId', 'rating']]
ratings_matrix = ratings.pivot_table('rating', index='userId', columns='movieId')
rating_movies = pd.merge(ratings, movies, on='movieId')
rating_matrix = rating_movies.pivot_table('rating', index='userId', columns='title')

In [11]:
#행렬분해. 시간 많이 걸리니까 SGD 반복 횟수는 200ghlaks
P, Q = matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01, r_lambda=0.01)
pred_matrix = np.dot(P, Q.T)

### iteration step :  0   rmse :  2.699204301709659
### iteration step :  50   rmse :  0.2614593544750243
### iteration step :  100   rmse :  0.18465739069584783
### iteration step :  150   rmse :  0.16336620703951366


In [12]:
### DF로 변환해 쉽게 보자

ratings_pred_matrix = pd.DataFrame(data=pred_matrix, index=ratings_matrix.index, columns=ratings_matrix.columns)
ratings_pred_matrix.head(3)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.219413,2.608552,4.131222,2.905101,3.626638,4.181860,4.133656,3.840058,3.670304,4.751578,...,3.032766,2.538513,3.400288,3.424991,3.083075,3.524228,3.017725,3.048294,3.065846,3.528559
2,3.797129,3.520305,4.203124,2.001279,3.178498,4.553509,3.131503,3.552015,3.417139,4.965818,...,2.728220,2.349227,2.981027,3.038971,2.732721,3.118977,2.691300,2.611148,2.687410,3.257123
3,1.783638,1.091696,1.584180,0.879563,-0.255868,2.633172,1.469916,2.951577,2.267331,0.963066,...,1.802815,1.481549,2.018660,1.934994,1.838333,2.064976,1.720691,1.666882,1.636386,2.065915


In [13]:
### 개인화된 영화 추천 해보자

def recomm_movie_by_userid(pred_df, userId, unseen_list, top_n=10):
    # 예측 평점 DF에서 사용자 아이디 인덱스와 unseen_list로 들어온 영화 칼럼명 추출 -> 가장 예측 평점 높은 순으로 정렬
    recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]
    return recomm_movies

def get_unseen_movies(ratings_matrix, userId):
    #userId로 입력받은 사용자의 모든 영화 정보를 추출해 Series로 반환
    #반환된 user_rating은 영화명을 인덱스로가지는 Series 객체임
    user_rating = ratings_matrix.loc[userId, :]
    #user_rating이 0보다 크면 기존에 관람한 영화임 -> 대신 인덱스 추출해 list 객체화
    already_seen = user_rating[user_rating>0].index.tolist()
    #모든 영화명을 리스트 객체로 만듦
    movies_list = ratings_matrix.columns.tolist()
    #리스트 내포로 alreay_seen 해당하는 영하는 리스트에서 제외함
    unseen_list = [movie for movie in movies_list if movie not in already_seen]
    
    return unseen_list

unseen_list = get_unseen_movies(ratings_matrix, 9)
recomm_movies = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)
recomm_movies = pd.DataFrame(data=recomm_movies.values, index=recomm_movies.index, columns=['pred_score'])
recomm_movies

,pred_score
movieId,
1291,5.459654
2064,5.290213
7153,5.231366
1230,5.204406
63082,4.994479
1148,4.956023
1278,4.942915
1287,4.922988
246,4.911752


* 9.6절의 결과와는 추천 결과가 사뭇 다름!